# 01d — Replogle 2022: Signature Scoring and GSEA

**Dataset:** ReplogleWeissman2022_rpe1 (scPerturb-harmonized)  
**Accession:** GSE181897  
**Phase:** 2  
**Date:** 2026-03-11  
**Objective:**
1. Over-Representation Analysis (ORA) on the DE gene lists produced in 01c
2. Gene Set Enrichment Analysis (GSEA) — preranked on full LFC profiles
3. Screen-wide pathway enrichment across all tested perturbations
4. Perturbation similarity matrix built from pathway enrichment scores + Leiden clustering
5. Single-cell gene set scoring for selected pathways (scanpy + decoupler)

## Table of Contents

1. [Setup](#setup)
2. [Load Data](#load) — DE results from 01c
3. [Background: From Genes to Pathways](#background)
   - 3a. What are gene sets?
   - 3b. Over-Representation Analysis (ORA) — the hypergeometric test
   - 3c. Gene Set Enrichment Analysis (GSEA) — the running-sum statistic
   - 3d. Single-cell gene set scoring
4. [Load Gene Set Collections](#genesets) — MSigDB hallmarks, KEGG, cardiac sets
5. [ORA: Single Perturbation Walk-through](#ora-single)
6. [GSEA Preranked: Single Perturbation Walk-through](#gsea-single)
7. [Screen-wide Pathway Enrichment](#screen-enrich) — all perturbations
8. [Perturbation Similarity Matrix](#similarity) — cosine on enrichment profiles
9. [Leiden Clustering of Pathway Programs](#leiden)
10. [Single-cell Gene Set Scoring](#sc-scoring)
11. [Save Results](#save)
12. [Summary](#summary)

<a id='setup'></a>
## 0. Setup

In [ ]:
import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scipy.sparse as sp
from scipy.spatial.distance import pdist, squareform
from scipy.stats import hypergeom
import gseapy as gp
import decoupler as dc
import boto3
import warnings
warnings.filterwarnings('ignore')

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False)
np.random.seed(42)

from pathlib import Path
PROC_DIR = Path('../../data/processed/scperturb')
RESULTS  = Path('../../results')
FIG_DIR  = RESULTS / 'figures'
TBL_DIR  = RESULTS / 'tables'
for d in [PROC_DIR, FIG_DIR, TBL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

S3_BUCKET = 'learn-perturb-seq'

# Thresholds — must match 01c
LFC_THRESHOLD = 0.5
FDR_THRESHOLD = 0.05

# Enrichment thresholds
ENRICH_FDR  = 0.05   # FDR cutoff for significant pathway enrichment
MIN_GENESET = 10     # min genes in a set after intersection with our background
MAX_GENESET = 500    # max genes in a set

print('Setup complete.')
print(f'gseapy version:    {gp.__version__}')
print(f'decoupler version: {dc.__version__}')

<a id='load'></a>
## 1. Load Data

We need two things from the previous notebooks:

1. **The processed AnnData** (`adata`) — for single-cell gene set scoring on individual cells.
2. **DE results from 01c** — the LFC matrix (perturbations × genes) and padj matrix.
   These are the inputs to ORA (significant gene lists) and GSEA (full ranked lists).

In [ ]:
# ---- Load processed AnnData ------------------------------------------------
scvi_path = PROC_DIR / 'ReplogleWeissman2022_rpe1_scvi.h5ad'
qc_path   = PROC_DIR / 'ReplogleWeissman2022_rpe1_qc.h5ad'
s3 = boto3.client('s3')

load_path = scvi_path if scvi_path.exists() else qc_path
if not load_path.exists():
    for candidate in [scvi_path, qc_path]:
        try:
            s3.download_file(S3_BUCKET, f'processed/scperturb/{candidate.name}', str(candidate))
            load_path = candidate
            print(f'Downloaded {candidate.name} from S3.')
            break
        except Exception:
            continue
    else:
        raise FileNotFoundError('Run 01a / 01b first.')

adata = sc.read_h5ad(load_path)
print(f'Loaded AnnData: {adata.shape}')

# ---- Load DE results from 01c ----------------------------------------------
lfc_path    = TBL_DIR / '01c_lfc_matrix.csv.gz'
padj_path   = TBL_DIR / '01c_padj_matrix.csv.gz'
de_sum_path = TBL_DIR / '01c_de_summary.csv'

for fpath in [lfc_path, padj_path, de_sum_path]:
    if not fpath.exists():
        try:
            s3.download_file(S3_BUCKET, f'results/tables/{fpath.name}', str(fpath))
            print(f'Downloaded {fpath.name} from S3.')
        except Exception:
            print(f'WARNING: {fpath.name} not found. Run 01c first.')

lfc_matrix  = pd.read_csv(lfc_path,  index_col=0, compression='gzip')
padj_matrix = pd.read_csv(padj_path, index_col=0, compression='gzip')
de_summary  = pd.read_csv(de_sum_path)

print(f'LFC matrix:  {lfc_matrix.shape}   (perturbations x genes)')
print(f'padj matrix: {padj_matrix.shape}')
print(f'DE summary:  {de_summary.shape}')

# Background gene universe = all genes tested in the DE
BACKGROUND = set(lfc_matrix.columns.tolist())
print(f'\nBackground gene universe (for ORA): {len(BACKGROUND):,} genes')

<a id='background'></a>
## 2. Background: From Genes to Pathways

The DE analysis in 01c produced lists of individual genes that change expression in each
perturbation. This is rich but hard to interpret directly: a single perturbation might affect
hundreds of genes, and comparing two perturbations at the gene level is noisy. Pathway-level
analysis solves this by asking: **which biological processes are perturbed?**

---

### 2a. What are gene sets?

A **gene set** is a curated collection of genes that share a biological property. Gene sets
are typically derived from:

| Source | Examples | Typical size |
|--------|----------|--------------|
| Canonical pathways | KEGG, Reactome, WikiPathways | 10–500 genes |
| Biological process (GO) | GO:0006096 (glycolysis) | 5–5000 genes |
| Transcription factor targets | JASPAR, ENCODE ChIP-seq | 20–2000 genes |
| **Hallmark gene sets (MSigDB H)** | HALLMARK_OXIDATIVE_PHOSPHORYLATION | 100–200 genes |
| Perturbation signatures | CGPS, L1000 | 50–500 genes |

**The Molecular Signatures Database (MSigDB)** is the most widely used compendium with
33,000+ gene sets in several collections:

- **H — Hallmarks:** 50 curated gene sets representing coherent biological states.
  These are the most interpretable and least redundant — ideal for overview analysis.
- **C2 — Curated pathways:** KEGG, Reactome, WikiPathways (thousands of sets).
- **C5 — Gene Ontology:** GO biological process / molecular function / cellular component.
- **C6 — Oncogenic signatures:** From cancer cell line perturbations.

---

### 2b. Over-Representation Analysis (ORA) — the hypergeometric test

ORA answers: **given a list of significant DE genes, is gene set G more enriched in
that list than expected by chance?**

**Setup:**
- $N$ = total background genes (all genes tested in DE, e.g. 15,000)
- $K$ = genes in set $G$ that are in the background (e.g., 100 for HALLMARK_GLYCOLYSIS)
- $n$ = total significant DE genes for this perturbation (e.g., 400)
- $k$ = significant DE genes that overlap with set $G$ (e.g., 20)

The **hypergeometric model** treats this as sampling without replacement — imagine a bag
with $N$ balls, $K$ of which are red (in the gene set). You draw $n$ balls. What is the
probability of drawing $k$ or more red balls?

$$P(X \geq k) = \sum_{i=k}^{\min(n,K)} \frac{\binom{K}{i} \binom{N-K}{n-i}}{\binom{N}{n}}$$

This is the one-tailed p-value. Multiple testing correction (Benjamini-Hochberg) is then
applied across all gene sets.

**Odds ratio** — how much more likely is a gene set member to appear in the DE list
compared to a random gene?
$$\text{OR} = \frac{k / n}{K / N} = \frac{\text{observed fraction}}{\text{expected fraction}}$$
OR = 3 means gene set members are 3× over-represented in the DE list.

**Key limitations of ORA:**
1. **Threshold-dependent:** Different LFC/FDR thresholds produce different enrichment results.
2. **Ignores effect size:** LFC = 10 and LFC = 0.6 contribute equally to the overlap count.
3. **Independence assumption:** The hypergeometric model ignores gene-gene correlations.

---

### 2c. Gene Set Enrichment Analysis (GSEA) — the running-sum statistic

GSEA uses the **entire ranked gene list** — no threshold needed. Genes are ranked by LFC,
and the algorithm asks: do genes in set $G$ cluster at the top or bottom of this ranking?

**Algorithm (Subramanian et al. 2005):**

1. **Rank all genes** by LFC (up-regulated at the top, down-regulated at the bottom).

2. **Walk down the ranked list.** Maintain a running sum:
   - **Increase** when a gene from set $G$ is encountered (weighted by |rank|)
   - **Decrease** when a gene not in $G$ is encountered (uniform step)

3. **The Enrichment Score (ES)** = maximum deviation of the running sum from zero.
   Positive ES → genes in $G$ are up-regulated. Negative ES → down-regulated.

4. **Significance:** ES is normalised by gene set size (NES), and a p-value is computed
   by permuting gene labels 1000 times to build the null distribution.

```
Ranked genes (most up → most down):
   G  .  .  G  .  G  G  .  .  .  G  .  .  .  .

Running sum:
    /\      /\
   /  \    /  \
--/    \--/    \--------

    ES = peak of the running sum
```

**GSEA advantages over ORA:**
- No threshold needed — uses all genes
- Accounts for effect size (larger |LFC| → larger contribution to running sum)
- Detects partial enrichment (only a subset of set genes need to be DE)

We use **preranked GSEA** (`gseapy.prerank`): genes are pre-ranked by LFC from 01c.

---

### 2d. Single-cell gene set scoring

ORA and GSEA operate on aggregated pseudobulk DE results. We can also score gene sets
at the **single-cell level** to quantify how much each individual cell activates a pathway.

**`scanpy.tl.score_genes` (Seurat-style):**
For each cell, compute the mean expression of the gene set minus the mean expression of
a random control set of the same size (matched for expression level). This gives a
z-score-like activity estimate per cell, visualisable on the UMAP.

**`decoupler.run_ulm` (univariate linear model):**
Regresses each gene set's weight vector against the cell's expression profile, producing
a t-statistic for every (cell, gene set) pair — more statistically principled than the
Seurat module score.

<a id='genesets'></a>
## 3. Load Gene Set Collections

We fetch three collections:

1. **MSigDB Hallmarks (H):** 50 gene sets representing well-defined biological states.
   Most interpretable and least redundant — ideal for screen overview.
2. **KEGG pathways:** Canonical metabolic and signaling pathways.
3. **Custom cardiovascular / metabolic sets:** Sarcomere, cardiac TFs, OXPHOS,
   ferroptosis — the primary biology of this project.

`gseapy.get_library()` fetches gene sets from MSigDB / Enrichr and returns
`{gene_set_name: [gene1, gene2, ...]}` dicts.

In [ ]:
# ---- Download MSigDB gene set collections via gseapy ----------------------
print('Fetching MSigDB Hallmarks...')
hallmarks = gp.get_library('MSigDB_Hallmark_2020', organism='Human')
print(f'  Hallmarks: {len(hallmarks)} gene sets')

print('Fetching KEGG pathways...')
kegg = gp.get_library('KEGG_2021_Human', organism='Human')
print(f'  KEGG: {len(kegg)} gene sets')

# ---- Custom cardiovascular / metabolic gene sets --------------------------
# Curated for this project's biological focus.
custom_sets = {
    'CARDIAC_SARCOMERE': [
        'MYH6', 'MYH7', 'MYL2', 'MYL3', 'TNNI3', 'TNNT2', 'TNNC1',
        'TPM1', 'TPM2', 'ACTC1', 'TTN', 'MYBPC3', 'TCAP', 'NEXN',
    ],
    'CARDIAC_TF_NETWORK': [
        'NKX2-5', 'GATA4', 'TBX5', 'MEF2C', 'HAND1', 'HAND2',
        'SMARCD3', 'TBX20', 'IRX4', 'ZBTB16',
    ],
    'MITOCHONDRIAL_OXPHOS': [
        'MT-ND1', 'MT-ND2', 'MT-ND3', 'MT-ND4', 'MT-ND5', 'MT-ND6',
        'MT-CO1', 'MT-CO2', 'MT-CO3', 'MT-CYB', 'MT-ATP6', 'MT-ATP8',
        'NDUFS1', 'NDUFS2', 'NDUFS3', 'NDUFV1', 'SDHB', 'SDHA',
        'UQCRC1', 'UQCRC2', 'COX4I1', 'ATP5F1A',
    ],
    'FERROPTOSIS': [
        'GPX4', 'SLC7A11', 'ACSL4', 'LPCAT3', 'ALOX15', 'ALOX5',
        'TFRC', 'SLC40A1', 'FTH1', 'FTL', 'HMOX1', 'NFE2L2',
        'G6PD', 'GCLC', 'GCLM', 'GSS',
    ],
    'UNFOLDED_PROTEIN_RESPONSE': [
        'ATF6', 'ERN1', 'EIF2AK3', 'XBP1', 'ATF4', 'DDIT3',
        'HSPA5', 'HERPUD1', 'DNAJB9', 'PPP1R15A',
    ],
    'CELL_CYCLE_G1_S': [
        'CDK4', 'CDK6', 'CCND1', 'CCND2', 'CCND3', 'RB1', 'E2F1',
        'E2F2', 'E2F3', 'CDKN1A', 'CDKN1B', 'CDKN2A',
    ],
}

print(f'\nCustom gene sets:')
for name, genes in custom_sets.items():
    print(f'  {name}: {len(genes)} genes')

# ---- Merge and filter by overlap with background gene universe -------------
all_genesets = {**hallmarks, **custom_sets}

filtered_genesets = {}
for name, genes in all_genesets.items():
    overlap = [g for g in genes if g in BACKGROUND]
    if MIN_GENESET <= len(overlap) <= MAX_GENESET:
        filtered_genesets[name] = overlap

print(f'\nGene sets after filtering ({MIN_GENESET}–{MAX_GENESET} genes in background):')
print(f'  {len(filtered_genesets)} sets retained out of {len(all_genesets)}')

# Build long-format net DataFrame for decoupler (source, target, weight)
net_rows = []
for gs_name, genes in filtered_genesets.items():
    for g in genes:
        net_rows.append({'source': gs_name, 'target': g, 'weight': 1.0})
net_df = pd.DataFrame(net_rows)
print(f'  decoupler net DataFrame: {len(net_df):,} rows')

<a id='ora-single'></a>
## 4. ORA: Single Perturbation Walk-through

We run ORA for the same walk-through perturbation as 01c, testing:
- **Up-regulated DE genes** (LFC > +0.5, FDR < 5%)
- **Down-regulated DE genes** (LFC < −0.5, FDR < 5%)

Testing up- and down-regulated sets separately is important because they carry
opposite biological meanings: up-enrichment means the pathway is *activated* by the
knockdown; down-enrichment means it is *suppressed*.

We implement ORA from scratch using `scipy.stats.hypergeom` to build intuition,
then visualise with a lollipop plot.

In [ ]:
# Pick the same walk-through perturbation as 01c (most DE genes)
TARGET = de_summary.sort_values('n_sig', ascending=False).iloc[0]['perturbation']
print(f'Walk-through perturbation: {TARGET}')
print(f'  n_sig DE genes: {de_summary[de_summary["perturbation"]==TARGET]["n_sig"].values[0]}')

lfc_target  = lfc_matrix.loc[TARGET].dropna()
padj_target = padj_matrix.loc[TARGET].reindex(lfc_target.index)

sig_up   = lfc_target.index[(padj_target < FDR_THRESHOLD) & (lfc_target >  LFC_THRESHOLD)].tolist()
sig_down = lfc_target.index[(padj_target < FDR_THRESHOLD) & (lfc_target < -LFC_THRESHOLD)].tolist()

print(f'\nDE gene lists: up={len(sig_up):,}  down={len(sig_down):,}')

In [ ]:
# ---- ORA from scratch: hypergeometric test --------------------------------

def run_ora(gene_list, gene_sets, background):
    """
    Over-representation analysis using the hypergeometric test.

    For each gene set G, tests whether G is enriched in gene_list more than
    expected from the background universe.

    Parameters
    ----------
    gene_list : list  — significant DE genes
    gene_sets : dict  — {name: [gene, ...]} (pre-filtered to background)
    background : set  — all tested genes

    Returns
    -------
    pd.DataFrame with p_value, padj (BH), odds_ratio per gene set
    """
    N = len(background)
    n = len([g for g in gene_list if g in background])
    rows = []

    for gs_name, gs_genes in gene_sets.items():
        gs_in_bg = [g for g in gs_genes if g in background]
        K = len(gs_in_bg)
        overlap = set(gene_list) & set(gs_in_bg)
        k = len(overlap)
        if K == 0 or n == 0:
            continue

        # P(X >= k) under Hypergeometric(N, K, n)
        # hypergeom.sf(k-1, M, n, N) = P(X >= k)
        # scipy parameterisation: sf(k, M, n, N) where
        #   M = population size, n = successes in population, N = draws
        p_val = hypergeom.sf(k - 1, N, K, n)

        expected   = n * K / N
        odds_ratio = (k / n) / (K / N) if K > 0 and n > 0 else np.nan

        rows.append({
            'gene_set':     gs_name,
            'N': N, 'K': K, 'n': n, 'k': k,
            'expected':     round(expected, 2),
            'odds_ratio':   round(odds_ratio, 3),
            'p_value':      p_val,
            'overlap_genes': ';'.join(sorted(overlap)),
        })

    df = pd.DataFrame(rows).sort_values('p_value').reset_index(drop=True)
    if len(df) == 0:
        return df

    # Benjamini-Hochberg FDR
    try:
        from scipy.stats import false_discovery_control
        df['padj'] = false_discovery_control(df['p_value'], method='bh')
    except Exception:
        m = len(df)
        df['padj'] = (df['p_value'] * m / (df.index + 1)).clip(upper=1.0)
        df['padj'] = df['padj'][::-1].cummin()[::-1]

    return df


ora_up   = run_ora(sig_up,   filtered_genesets, BACKGROUND)
ora_down = run_ora(sig_down, filtered_genesets, BACKGROUND)

sig_up_enr   = ora_up  [ora_up  ['padj'] < ENRICH_FDR].sort_values('odds_ratio', ascending=False)
sig_down_enr = ora_down[ora_down['padj'] < ENRICH_FDR].sort_values('odds_ratio', ascending=False)

print(f'ORA for {TARGET}:')
print(f'  Enriched in UP-regulated genes:   {len(sig_up_enr)}')
print(f'  Enriched in DOWN-regulated genes: {len(sig_down_enr)}')

print('\nTop UP-enriched gene sets:')
if len(sig_up_enr):
    print(sig_up_enr[['gene_set','K','k','expected','odds_ratio','padj']].head(8).to_string(index=False))

print('\nTop DOWN-enriched gene sets:')
if len(sig_down_enr):
    print(sig_down_enr[['gene_set','K','k','expected','odds_ratio','padj']].head(8).to_string(index=False))

In [ ]:
# ---- Lollipop plot: ORA enrichment results --------------------------------
top_n = 10
up_top   = sig_up_enr.head(top_n).copy().assign(direction='Up')
down_top = sig_down_enr.head(top_n).copy().assign(direction='Down')
combined = pd.concat([up_top, down_top], ignore_index=True)

if len(combined) == 0:
    print('No significant ORA results to plot.')
else:
    combined['log2_or'] = np.log2(combined['odds_ratio'].clip(lower=0.01))
    combined = combined.sort_values('log2_or')
    colors   = combined['direction'].map({'Up': 'tomato', 'Down': 'steelblue'})

    fig, ax = plt.subplots(figsize=(10, max(4, len(combined) * 0.4)))
    ax.barh(range(len(combined)), combined['log2_or'].values, color=colors, alpha=0.75, height=0.7)
    ax.set_yticks(range(len(combined)))
    ax.set_yticklabels(combined['gene_set'].str[:55].values, fontsize=8)
    ax.axvline(0, color='black', lw=0.6)
    ax.set_xlabel('log₂(Odds Ratio)')
    ax.set_title(
        f'ORA: {TARGET} CRISPRi\n'
        f'Top {top_n} enriched pathways (up / down-regulated DE genes)',
    )
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(color='tomato',    label='Up-regulated DE genes'),
        Patch(color='steelblue', label='Down-regulated DE genes'),
    ], fontsize=8)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'01d_ora_{TARGET}.pdf', bbox_inches='tight')
    plt.show()

<a id='gsea-single'></a>
## 5. GSEA Preranked: Single Perturbation Walk-through

We rank all genes by their LFC from the pseudobulk DE, then run `gseapy.prerank()`.
No significance threshold is used — every gene participates in the ranking.

**Ranking statistic:** signed LFC.
- Genes at the top: up-regulated in the perturbation vs. control
- Genes at the bottom: down-regulated

An alternative statistic is the **signed −log10(p-value)**:
$$\text{rank}_g = \text{sign}(\text{LFC}_g) \times {-}\log_{10}(p_g)$$
This prioritises highly significant genes but can down-weight genes with large LFC
but moderate p-values. We use plain LFC here for consistency with the similarity matrix.

**Output:** A Normalized Enrichment Score (NES) per gene set:
- NES > 0: set members tend to be up-regulated
- NES < 0: set members tend to be down-regulated
- |NES| > 1.5 and FDR < 5%: typically considered significant

In [ ]:
# ---- GSEA preranked for the walk-through perturbation ---------------------
rank_series = lfc_target.dropna().sort_values(ascending=False)

print(f'Ranked gene list: {len(rank_series):,} genes')
print(f'  Top 5 up:   {rank_series.head(5).to_dict()}')
print(f'  Top 5 down: {rank_series.tail(5).to_dict()}')
print()

gsea_result = gp.prerank(
    rnk=rank_series,
    gene_sets=filtered_genesets,
    min_size=MIN_GENESET,
    max_size=MAX_GENESET,
    permutation_num=1000,
    outdir=None,
    seed=42,
    verbose=False,
)

gsea_df = (
    gsea_result.res2d
    .rename(columns={'Term': 'gene_set', 'NOM p-val': 'pvalue', 'FDR q-val': 'padj'})
    .sort_values('padj')
)

sig_gsea = gsea_df[gsea_df['padj'] < ENRICH_FDR]
print(f'Significant GSEA results (FDR < {ENRICH_FDR}): {len(sig_gsea)}')
print()
print(sig_gsea[['gene_set', 'NES', 'pvalue', 'padj']].head(15).to_string(index=False))

In [ ]:
# ---- GSEA dot plot: top up + top down by NES --------------------------------
plot_df = gsea_df.copy()
plot_df['NES_num'] = pd.to_numeric(plot_df['NES'], errors='coerce')
plot_df['neg_logp'] = -np.log10(plot_df['padj'].clip(lower=1e-30))
plot_df = plot_df.dropna(subset=['NES_num'])

top_n = 10
plot_sub = pd.concat([
    plot_df.nsmallest(top_n, 'NES_num'),
    plot_df.nlargest(top_n, 'NES_num'),
]).sort_values('NES_num')

fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(
    plot_sub['NES_num'],
    range(len(plot_sub)),
    c=plot_sub['NES_num'], cmap='RdBu_r', vmin=-3, vmax=3,
    s=plot_sub['neg_logp'] * 20 + 10,
    edgecolors='grey', linewidths=0.3, zorder=3,
)
plt.colorbar(scatter, ax=ax, label='NES', shrink=0.6)
ax.set_yticks(range(len(plot_sub)))
ax.set_yticklabels(plot_sub['gene_set'].str[:60].values, fontsize=7)
ax.axvline(0, color='black', lw=0.5)
ax.set_xlabel('Normalized Enrichment Score (NES)')
ax.set_title(
    f'GSEA preranked: {TARGET} CRISPRi\n'
    f'Top {top_n} up / {top_n} down  |  dot size ∝ −log₁₀(FDR)'
)
plt.tight_layout()
plt.savefig(FIG_DIR / f'01d_gsea_{TARGET}.pdf', bbox_inches='tight')
plt.show()

# Running-sum enrichment plot for the top gene set
top_gs_name = plot_df.loc[plot_df['NES_num'].idxmax(), 'gene_set']
try:
    gsea_result.plot(terms=top_gs_name, show_ranking=True, figsize=(8, 4))
    plt.suptitle(f'Running-sum enrichment: {top_gs_name}\n({TARGET} CRISPRi)', y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'01d_gsea_running_sum_{TARGET}.pdf', bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'Running-sum plot skipped: {e}')

<a id='screen-enrich'></a>
## 6. Screen-wide Pathway Enrichment

We run preranked GSEA for every perturbation in the LFC matrix. The output is a
**NES matrix** (perturbations × gene sets) — the foundation for the pathway-level
similarity matrix in Section 7.

Using NES rather than raw LFC for the similarity matrix offers a key advantage:
the NES matrix is **lower-dimensional** (50–200 pathways vs. ~15,000 genes) and
**less noisy** (per-gene estimates are aggregated within each pathway).

In [ ]:
# ---- Screen-wide GSEA: build NES matrix ------------------------------------
MAX_PERTS = min(len(lfc_matrix), 300)   # cap for tutorial speed
perts_list = lfc_matrix.index.tolist()[:MAX_PERTS]

print(f'Running preranked GSEA for {len(perts_list)} perturbations...')
print(f'Gene sets tested per perturbation: {len(filtered_genesets)}')
print()

all_nes      = {}   # pert -> {gene_set: NES}
all_gsea_padj = {}  # pert -> {gene_set: FDR}
gsea_failed  = []

for i, pert in enumerate(perts_list):
    rank_s = lfc_matrix.loc[pert].dropna().sort_values(ascending=False)
    if len(rank_s) < 50:
        gsea_failed.append(pert)
        continue
    try:
        res = gp.prerank(
            rnk=rank_s,
            gene_sets=filtered_genesets,
            min_size=MIN_GENESET,
            max_size=MAX_GENESET,
            permutation_num=200,   # fewer for speed in screen mode
            outdir=None, seed=42, verbose=False,
        )
        all_nes[pert]       = res.res2d.set_index('Term')['NES'].to_dict()
        all_gsea_padj[pert] = res.res2d.set_index('Term')['FDR q-val'].to_dict()
    except Exception:
        gsea_failed.append(pert)

    if (i + 1) % 50 == 0 or (i + 1) == len(perts_list):
        print(f'  [{i+1:4d}/{len(perts_list)}] done  (failed: {len(gsea_failed)})')

nes_matrix       = pd.DataFrame(all_nes).T.astype(float)
gsea_padj_matrix = pd.DataFrame(all_gsea_padj).T.astype(float)

print(f'\nNES matrix: {nes_matrix.shape}   (perturbations x pathways)')
print(f'NES range: [{nes_matrix.values[~np.isnan(nes_matrix.values)].min():.2f},'
      f' {nes_matrix.values[~np.isnan(nes_matrix.values)].max():.2f}]')

In [ ]:
# ---- Screen-wide summary: most frequently enriched gene sets ---------------
sig_mask = gsea_padj_matrix < ENRICH_FDR

gs_summary = pd.DataFrame({
    'n_sig_perts': sig_mask.sum(axis=0),
    'mean_nes':    nes_matrix.mean(axis=0),
    'max_nes':     nes_matrix.max(axis=0),
    'min_nes':     nes_matrix.min(axis=0),
}).sort_values('n_sig_perts', ascending=False)

print('Top 20 most frequently enriched gene sets:')
print(gs_summary.head(20).to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_gs = gs_summary.head(20)
axes[0].barh(range(len(top_gs)), top_gs['n_sig_perts'].values[::-1], color='steelblue')
axes[0].set_yticks(range(len(top_gs)))
axes[0].set_yticklabels(top_gs.index[::-1], fontsize=8)
axes[0].set_xlabel('# perturbations with significant enrichment (FDR < 5%)')
axes[0].set_title('Most globally enriched pathways')

# NES heatmap: top 20 pathways x top 30 perturbations
top_gs_names  = gs_summary.head(20).index.tolist()
top_pert_names = de_summary.nlargest(30, 'n_sig')['perturbation'].tolist()
top_pert_names = [p for p in top_pert_names if p in nes_matrix.index]

if len(top_pert_names) >= 2 and len(top_gs_names) >= 2:
    nes_sub = nes_matrix.loc[top_pert_names, top_gs_names].fillna(0)
    sns.heatmap(
        nes_sub, ax=axes[1],
        cmap='RdBu_r', center=0, vmin=-3, vmax=3,
        xticklabels=[n[:25] for n in top_gs_names],
        yticklabels=top_pert_names,
        cbar_kws={'label': 'NES'},
    )
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=7)
    axes[1].set_yticklabels(axes[1].get_yticklabels(), fontsize=7)
    axes[1].set_title('NES: top 30 perturbations × top 20 pathways')

plt.suptitle('Screen-wide GSEA summary', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / '01d_screen_gsea_summary.pdf', bbox_inches='tight')
plt.show()

<a id='similarity'></a>
## 7. Perturbation Similarity Matrix

### What is a perturbation similarity matrix?

A **perturbation similarity matrix** is a square matrix $S \in \mathbb{R}^{P \times P}$
where $P$ is the number of perturbations. Entry $S_{ij}$ quantifies how similar the
downstream transcriptional response of perturbation $i$ is to perturbation $j$.

High similarity between $i$ and $j$ implies one or more of:
- Genes $i$ and $j$ act in the **same pathway** (knocking either one propagates the same
  signal disruption downstream)
- Gene $j$ is a **direct transcriptional target** of gene $i$
- Genes $i$ and $j$ are **co-regulated** or physically interact in the same complex

---

### Gene-level vs. pathway-level representation

We built a gene-level similarity matrix in 01c from raw LFC profiles. Here we build
a pathway-level one from NES profiles. These are complementary views:

| Property | Gene-level (LFC, ~15,000 dim) | Pathway-level (NES, 50–200 dim) |
|----------|------------------------------|---------------------------------|
| Noise | Higher — individual gene estimates are noisy | Lower — noise averaged within pathways |
| Dimensionality | High → curse of dimensionality | Low → better k-NN |
| Interpretation | Exact genes changed | Biological processes altered |
| Sensitivity | Unique gene-level effects | Misses one-gene effects |
| Clustering quality | Noisier groupings | More biologically coherent |

**Example:** Two perturbations that knock down different subunits of Complex I (OXPHOS).
Gene-level similarity might be modest (they share some downstream targets but each has
unique neighbours). Pathway-level similarity is high (both show NES ≈ −2 for
`HALLMARK_OXIDATIVE_PHOSPHORYLATION`), correctly identifying them as OXPHOS perturbations.

---

### Cosine similarity: the standard metric for profile comparison

For each perturbation $i$ we have an NES profile vector
$\mathbf{v}_i \in \mathbb{R}^d$ where $d$ = number of gene sets.

**Cosine similarity:**
$$S_{\cos}(i, j) = \frac{\mathbf{v}_i \cdot \mathbf{v}_j}{\|\mathbf{v}_i\| \, \|\mathbf{v}_j\|}$$

This is the cosine of the angle between the two profile vectors:
- $S_{\cos} = +1.0$: same pathways activated / suppressed in the same direction
- $S_{\cos} = 0.0$: no shared pathway signal (orthogonal profiles)
- $S_{\cos} = -1.0$: opposite effects (what activates pathways in $i$ suppresses them in $j$)

**Why cosine instead of Euclidean?**

Euclidean distance is sensitive to the *magnitude* of the effect:
$$d_{\text{Euc}}(i, j) = \|\mathbf{v}_i - \mathbf{v}_j\|_2$$

A perturbation with uniformly weak NES scores (subtle effect) and one with strong NES
scores but the same *pattern* will appear far apart in Euclidean space, even though
they regulate the same pathways. Cosine normalises by vector magnitude, capturing
**direction** (which pathways, which way) independently of **strength**.

In Perturb-seq this matters because:
- Essential genes often have very strong effects (large |NES|)
- Modulators have mild effects (small |NES|)
- We want to know if an essential and a modulator affect the same pathway

---

### Interpreting the heatmap

**Block structure:** Groups of perturbations with high within-group similarity form
rectangular blocks. Each block represents a shared transcriptional program — perturbations
that disrupt the same biological process.

**Anti-correlated blocks:** If block A and block B have negative off-diagonal similarity,
they regulate opposing processes (e.g., pro-apoptotic vs. anti-apoptotic).

**Diagonal = 1:** Always. The interesting signal is in the off-diagonal entries.

**Relationship to network inference:** The similarity matrix is a weighted gene-gene
interaction graph (edge = similarity value). Leiden clustering on this graph recovers
functional modules — analogous to STRING protein networks but derived from the data
rather than the literature.

---

### Practical details

- **NaN handling:** Missing NES values (gene sets with too few genes) are filled with 0
  before computing cosine similarity. Zero means "no enrichment signal" which is appropriate.
- **Pathway selection:** We use all filtered gene sets (MIN_GENESET ≤ genes ≤ MAX_GENESET).
  Including very small or very large gene sets can distort the similarity.
- **Normalisation:** Cosine similarity already normalises for vector magnitude, so no
  additional Z-scoring of the NES matrix is needed before computing similarities.

In [ ]:
# ---- Build pathway-level perturbation similarity matrix --------------------
nes_filled = nes_matrix.fillna(0)

print(f'NES matrix for similarity: {nes_filled.shape}')
print(f'  {nes_filled.shape[0]} perturbations  x  {nes_filled.shape[1]} gene sets')

# Pairwise cosine distance, then convert to similarity
cosine_dist_nes = squareform(pdist(nes_filled.values, metric='cosine'))
cosine_sim_nes  = 1 - cosine_dist_nes

sim_nes_df = pd.DataFrame(cosine_sim_nes, index=nes_filled.index, columns=nes_filled.index)

# Off-diagonal statistics
upper_tri = cosine_sim_nes[np.triu_indices_from(cosine_sim_nes, k=1)]
print(f'\nOff-diagonal similarity distribution:')
print(f'  Mean:         {upper_tri.mean():.3f}')
print(f'  Std:          {upper_tri.std():.3f}')
print(f'  Median:       {np.median(upper_tri):.3f}')
print(f'  > 0.5:        {(upper_tri > 0.5).mean():.1%} of pairs')
print(f'  > 0.7:        {(upper_tri > 0.7).mean():.1%} of pairs')
print(f'  < 0 (anti):   {(upper_tri < 0.0).mean():.1%} of pairs')

# Distribution
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(upper_tri, bins=60, color='steelblue', edgecolor='none')
ax.axvline(0,   color='black', lw=0.7)
ax.axvline(0.5, color='tomato', ls='--', lw=1, label='similarity = 0.5')
ax.set_xlabel('Pairwise cosine similarity (NES profiles)')
ax.set_ylabel('Number of perturbation pairs')
ax.set_title(
    f'Pathway-level perturbation similarity distribution\n'
    f'{nes_filled.shape[0]} perturbations, {nes_filled.shape[1]} gene sets'
)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / '01d_similarity_distribution.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- Clustermap: pathway-level similarity for top perturbations ------------
N_TOP = min(50, len(sim_nes_df))
top_perts = de_summary.nlargest(N_TOP, 'n_sig')['perturbation'].tolist()
top_perts = [p for p in top_perts if p in sim_nes_df.index]

sim_top = sim_nes_df.loc[top_perts, top_perts]

cg = sns.clustermap(
    sim_top,
    cmap='RdBu_r', center=0, vmin=-1.0, vmax=1.0,
    xticklabels=True, yticklabels=True,
    linewidths=0, figsize=(14, 12),
    dendrogram_ratio=0.12,
    cbar_pos=(0.02, 0.82, 0.03, 0.15),
    cbar_kws={'label': 'Cosine similarity (NES)'},
)
cg.ax_heatmap.set_xticklabels(cg.ax_heatmap.get_xticklabels(), fontsize=7, rotation=90)
cg.ax_heatmap.set_yticklabels(cg.ax_heatmap.get_yticklabels(), fontsize=7)
cg.figure.suptitle(
    f'Pathway-level perturbation similarity (cosine on NES profiles)\n'
    f'Top {len(top_perts)} perturbations by DE gene count',
    y=1.01, fontsize=11,
)
cg.figure.savefig(FIG_DIR / '01d_pathway_similarity_heatmap.pdf', bbox_inches='tight')
plt.show()

# ---- Compare to gene-level similarity from 01c ----------------------------
lfc_sim_path = TBL_DIR / '01c_cosine_similarity.csv.gz'
if lfc_sim_path.exists():
    lfc_sim_df   = pd.read_csv(lfc_sim_path, index_col=0, compression='gzip')
    shared_perts = [p for p in top_perts if p in lfc_sim_df.index]

    if len(shared_perts) >= 10:
        lfc_vals = lfc_sim_df.loc[shared_perts, shared_perts].values
        nes_vals = sim_nes_df.loc[shared_perts, shared_perts].values
        idx = np.triu_indices_from(lfc_vals, k=1)
        r   = np.corrcoef(lfc_vals[idx], nes_vals[idx])[0, 1]

        fig, ax = plt.subplots(figsize=(6, 5))
        ax.scatter(lfc_vals[idx], nes_vals[idx], s=4, alpha=0.3,
                   color='steelblue', rasterized=True)
        ax.plot([-1, 1], [-1, 1], 'r--', lw=0.8)
        ax.set_xlabel('Gene-level cosine similarity (LFC)')
        ax.set_ylabel('Pathway-level cosine similarity (NES)')
        ax.set_title(f'Gene-level vs. pathway-level similarity  r = {r:.3f}\n'
                     f'({len(shared_perts)} perturbations)')
        plt.tight_layout()
        plt.savefig(FIG_DIR / '01d_similarity_comparison.pdf', bbox_inches='tight')
        plt.show()
        print(f'Pearson r (gene vs. pathway similarity) = {r:.3f}')
else:
    print('01c cosine similarity file not found — skipping comparison.')

<a id='leiden'></a>
## 8. Leiden Clustering of Pathway Programs

We cluster perturbations by their NES profiles using the same Leiden approach as 01c,
but now at the pathway level (50–200 dimensions instead of ~15,000).

Lower dimensionality benefits the k-NN graph construction: in high dimensions, all
pairwise distances tend to converge (the *curse of dimensionality*), making the
k-NN graph poorly discriminative. At 50–200 dimensions this is much less of a problem.

After clustering, we **characterise each cluster by its mean NES profile** — the dominant
pathways activated or suppressed across the cluster members. This gives a one-line
biological description of what each cluster represents.

In [ ]:
# ---- AnnData of perturbations (NES profiles as 'expression') ---------------
adata_pert = ad.AnnData(
    X=nes_filled.values,
    obs=pd.DataFrame(index=nes_filled.index),
    var=pd.DataFrame(index=nes_filled.columns),
)
adata_pert.obs = adata_pert.obs.join(
    de_summary.set_index('perturbation')[['n_cells', 'n_sig', 'n_up', 'n_down']],
    how='left',
)

# PCA → k-NN → Leiden → UMAP
N_PCA = min(20, nes_filled.shape[0] - 2, nes_filled.shape[1] - 2)
sc.tl.pca(adata_pert, n_comps=N_PCA)
sc.pp.neighbors(adata_pert, n_neighbors=15, use_rep='X_pca')
sc.tl.leiden(adata_pert, resolution=0.6, key_added='leiden_pathway', random_state=42)
sc.tl.umap(adata_pert, random_state=42)

n_clusters = adata_pert.obs['leiden_pathway'].nunique()
print(f'Leiden clustering on NES profiles: {n_clusters} clusters')
print()

# Characterise each cluster: mean NES profile
cluster_profiles = {}
for cl in sorted(adata_pert.obs['leiden_pathway'].unique(), key=int):
    members  = adata_pert.obs[adata_pert.obs['leiden_pathway'] == cl].index.tolist()
    mean_nes = nes_filled.loc[members].mean(axis=0)
    cluster_profiles[cl] = mean_nes
    top_up   = mean_nes.nlargest(3).index.tolist()
    top_down = mean_nes.nsmallest(3).index.tolist()
    print(f'Cluster {cl}  ({len(members)} perturbations):')
    print(f'  Top activated:   {top_up}')
    print(f'  Top suppressed:  {top_down}')
    if len(members) <= 8:
        print(f'  Members: {members}')
    else:
        top5 = adata_pert.obs.loc[members].nlargest(5, 'n_sig').index.tolist()
        print(f'  Top 5 by n_sig:  {top5}')
    print()

In [ ]:
# ---- UMAP + cluster mean NES heatmap ----------------------------------------
fig = plt.figure(figsize=(18, 10))
gs_fig = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

ax0 = fig.add_subplot(gs_fig[0, 0])
ax1 = fig.add_subplot(gs_fig[0, 1])
ax2 = fig.add_subplot(gs_fig[0, 2])

sc.pl.umap(adata_pert, color='leiden_pathway', ax=ax0, show=False,
           title='Perturbation clusters (NES profiles)', legend_loc='right margin', s=30)
sc.pl.umap(adata_pert, color='n_sig', ax=ax1, show=False,
           title='# significant DE genes', color_map='viridis', s=30)
sc.pl.umap(adata_pert, color='n_cells', ax=ax2, show=False,
           title='Cells per perturbation', color_map='Blues', s=30)

# Mean NES heatmap per cluster: top 20 most variable pathways
ax_hm = fig.add_subplot(gs_fig[1, :])
cluster_mean_df = pd.DataFrame(cluster_profiles).T
pathway_var     = cluster_mean_df.var(axis=0).nlargest(20).index
hm_data         = cluster_mean_df[pathway_var]

im = ax_hm.imshow(hm_data.values, cmap='RdBu_r', aspect='auto', vmin=-2, vmax=2)
ax_hm.set_xticks(range(len(hm_data.columns)))
ax_hm.set_xticklabels([n[:35] for n in hm_data.columns], rotation=45, ha='right', fontsize=7)
ax_hm.set_yticks(range(len(hm_data.index)))
ax_hm.set_yticklabels([f'Cluster {c}' for c in hm_data.index], fontsize=8)
ax_hm.set_title('Mean NES per cluster — top 20 most variable pathways', fontsize=9)
plt.colorbar(im, ax=ax_hm, label='Mean NES', shrink=0.6, pad=0.01)

plt.suptitle('Pathway-program clustering — Leiden on NES profiles', fontsize=12, y=1.01)
plt.savefig(FIG_DIR / '01d_pathway_leiden_umap.pdf', bbox_inches='tight')
plt.show()

<a id='sc-scoring'></a>
## 9. Single-cell Gene Set Scoring

ORA and GSEA operate on pseudobulk DE results — they describe pathway enrichment
averaged over all cells in a perturbation group. Single-cell scoring is complementary:
it assigns a **pathway activity score to each individual cell**.

This lets us:
- Visualise pathway activity on the UMAP
- Detect within-perturbation heterogeneity (not all cells respond equally)
- Correlate pathway activity with technical covariates (UMI count, guide coverage)

### `scanpy.tl.score_genes` (Seurat module score)
For gene set $G$ in cell $c$:
$$\text{score}_c = \text{mean}_{g \in G}(x_{cg}) - \text{mean}_{g' \in G_{\text{ctrl}}}(x_{cg'})$$
where $G_{\text{ctrl}}$ is a random control gene set of the same size as $G$, matched
for mean expression level. The subtraction corrects for differences in library size.

### `decoupler.run_ulm` (univariate linear model)
For each (cell $c$, gene set $G$): regress the cell's expression profile onto the gene
set weight vector, producing a t-statistic. More statistically principled than the
Seurat module score — can produce per-cell p-values.

In [ ]:
# ---- scanpy module scores: per-cell pathway activity ----------------------
score_sets = {
    'OXPHOS':       custom_sets['MITOCHONDRIAL_OXPHOS'],
    'UPR':          custom_sets['UNFOLDED_PROTEIN_RESPONSE'],
    'Ferroptosis':  custom_sets['FERROPTOSIS'],
    'Cell_Cycle':   custom_sets['CELL_CYCLE_G1_S'],
}
# Add top hallmark sets if present
for hl in ['HALLMARK_MYC_TARGETS_V1', 'HALLMARK_OXIDATIVE_PHOSPHORYLATION',
            'HALLMARK_GLYCOLYSIS', 'HALLMARK_P53_PATHWAY']:
    if hl in filtered_genesets:
        score_sets[hl.replace('HALLMARK_', '')] = filtered_genesets[hl]

print('Scoring gene sets per cell (scanpy.tl.score_genes):')
for name, genes in score_sets.items():
    overlap = [g for g in genes if g in adata.var_names]
    if len(overlap) < 5:
        print(f'  {name}: only {len(overlap)} genes found in adata — skipping')
        continue
    sc.tl.score_genes(adata, gene_list=overlap, score_name=f'score_{name}', random_state=42)
    print(f'  {name}: scored ({len(overlap)} genes)')

score_keys = [k for k in adata.obs.columns if k.startswith('score_')]
print(f'\nScore keys in adata.obs: {score_keys}')

In [ ]:
# ---- UMAP coloured by gene set scores --------------------------------------
umap_key = 'X_umap_scvi' if 'X_umap_scvi' in adata.obsm else 'X_umap'

if len(score_keys) == 0:
    print('No scores available.')
elif umap_key not in adata.obsm and 'X_umap' not in adata.obsm:
    print('No UMAP found — run 01b first.')
else:
    if umap_key != 'X_umap':
        adata.obsm['_umap_backup'] = adata.obsm.get('X_umap')
        adata.obsm['X_umap'] = adata.obsm[umap_key]

    n_cols = min(3, len(score_keys))
    n_rows = int(np.ceil(len(score_keys) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
    axes = np.array(axes).flatten()

    for i, key in enumerate(score_keys):
        sc.pl.umap(adata, color=key, ax=axes[i], show=False,
                   color_map='RdBu_r', vcenter=0,
                   title=key.replace('score_', '').replace('_', ' '))
    for j in range(len(score_keys), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Single-cell gene set scores on UMAP', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(FIG_DIR / '01d_sc_gene_set_scores_umap.pdf', bbox_inches='tight')
    plt.show()

    if umap_key != 'X_umap' and adata.obsm.get('_umap_backup') is not None:
        adata.obsm['X_umap'] = adata.obsm.pop('_umap_backup')

In [ ]:
# ---- Violin: score distribution across top perturbations ------------------
if len(score_keys) > 0:
    score_key  = score_keys[0]
    score_name = score_key.replace('score_', '')

    top_perts_score = (
        adata.obs.groupby('perturbation')[score_key]
        .mean().abs().sort_values(ascending=False).head(15).index.tolist()
    )
    if 'control' not in top_perts_score:
        top_perts_score.append('control')

    adata_sub = adata[adata.obs['perturbation'].isin(top_perts_score)].copy()
    order = (
        adata_sub.obs.groupby('perturbation')[score_key]
        .mean().sort_values(ascending=False).index.tolist()
    )

    fig, ax = plt.subplots(figsize=(14, 5))
    sns.violinplot(
        data=adata_sub.obs, x='perturbation', y=score_key, order=order,
        ax=ax, palette='Set2', scale='width', inner='quartile', linewidth=0.5,
    )
    ax.axhline(0, color='black', lw=0.5)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax.set_xlabel('Perturbation')
    ax.set_ylabel(f'{score_name} score (per cell)')
    ax.set_title(f'Per-cell {score_name} score — top 15 perturbations by mean |score|')
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'01d_sc_score_violin_{score_name}.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# ---- decoupler ULM: per-cell activity for all gene sets --------------------
# dc.run_ulm fits a univariate linear model per (cell, gene set) pair.
# Result stored in adata.obsm['ulm_estimate'] and adata.obsm['ulm_pvals']

print('Running decoupler ULM on all gene sets...')
dc.run_ulm(
    mat=adata, net=net_df,
    source='source', target='target', weight='weight',
    verbose=False,
)

ulm_acts = dc.get_acts(adata, obsm_key='ulm_estimate')
print(f'ULM activity matrix: {ulm_acts.shape}   (cells x gene sets)')

act_var = ulm_acts.to_df().var(axis=0).sort_values(ascending=False)
print(f'\nTop 10 most variable gene set activities:')
print(act_var.head(10).to_string())

# Plot on UMAP
top_var_sets = act_var.head(4).index.tolist()
umap_key = 'X_umap_scvi' if 'X_umap_scvi' in adata.obsm else 'X_umap'
if umap_key in adata.obsm and umap_key != 'X_umap':
    adata.obsm['X_umap'] = adata.obsm[umap_key]

if 'X_umap' in adata.obsm and len(top_var_sets) > 0:
    sc.pl.umap(
        ulm_acts, color=top_var_sets, ncols=2,
        color_map='RdBu_r', vcenter=0,
        title=[f'ULM: {s[:40]}' for s in top_var_sets],
        show=False,
    )
    plt.suptitle('decoupler ULM gene set activity — top 4 most variable', y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / '01d_ulm_activity_umap.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# <a id='save'></a>
# === 10. Save Results ======================================================

nes_path     = TBL_DIR / '01d_nes_matrix.csv.gz'
sim_path     = TBL_DIR / '01d_pathway_similarity.csv.gz'
cluster_path = TBL_DIR / '01d_perturbation_pathway_clusters.csv'
ora_path     = TBL_DIR / f'01d_ora_{TARGET}.csv'
gsea_path    = TBL_DIR / f'01d_gsea_{TARGET}.csv'
scored_path  = PROC_DIR / 'ReplogleWeissman2022_rpe1_scored.h5ad'

nes_matrix.to_csv(nes_path, compression='gzip')
sim_nes_df.to_csv(sim_path, compression='gzip')
adata_pert.obs[['leiden_pathway', 'n_sig', 'n_cells']].to_csv(cluster_path)
pd.concat([ora_up.assign(direction='up'), ora_down.assign(direction='down')],
          ignore_index=True).to_csv(ora_path, index=False)
gsea_df.to_csv(gsea_path, index=False)
adata.write_h5ad(scored_path)

print('Saved:')
for p in [nes_path, sim_path, cluster_path, ora_path, gsea_path, scored_path]:
    print(f'  {p}')

# S3 upload
s3 = boto3.client('s3')
for fpath in [nes_path, sim_path, cluster_path, ora_path, gsea_path]:
    try:
        s3.upload_file(str(fpath), S3_BUCKET, f'results/tables/{fpath.name}')
    except Exception:
        pass
try:
    s3.upload_file(str(scored_path), S3_BUCKET, f'processed/scperturb/{scored_path.name}')
except Exception:
    pass
print('S3 upload attempted.')

In [ ]:
# <a id='summary'></a>
print('=' * 65)
print('01d SIGNATURE SCORING — SUMMARY')
print('=' * 65)
print()

checks = [
    ('MSigDB Hallmarks loaded',                              len(hallmarks) > 0),
    (f'Gene sets after filtering: {len(filtered_genesets)}', len(filtered_genesets) > 0),
    (f'ORA run for {TARGET}',                                len(ora_up) > 0 or len(ora_down) > 0),
    (f'GSEA preranked run for {TARGET}',                     len(gsea_df) > 0),
    (f'Screen GSEA: {len(nes_matrix)} perturbations',        len(nes_matrix) > 0),
    (f'NES matrix: {nes_matrix.shape}',                      nes_matrix.shape[0] > 0),
    (f'Pathway similarity: {sim_nes_df.shape}',              sim_nes_df.shape[0] > 0),
    (f'Leiden: {n_clusters} pathway clusters',               n_clusters > 0),
    (f'Single-cell scores: {len(score_keys)} sets',          len(score_keys) > 0),
    ('decoupler ULM run',                                    'ulm_estimate' in adata.obsm),
    ('NES matrix saved',                                     nes_path.exists()),
]

all_pass = True
for check, result in checks:
    status = 'v' if result else 'X'
    print(f'  [{status}]  {check}')
    if not result:
        all_pass = False

print()
print(f'  Walk-through: {TARGET}')
print(f'    ORA enriched (up):   {len(sig_up_enr)}')
print(f'    ORA enriched (down): {len(sig_down_enr)}')
print(f'    GSEA significant:    {len(sig_gsea)}')
top_gsea_name = gsea_df.iloc[0]["gene_set"] if len(gsea_df) else "N/A"
print(f'    Top GSEA pathway:    {top_gsea_name}')
print()
print(f'  Screen NES matrix: {nes_matrix.shape}')
print(f'  Pathway clusters:  {n_clusters}')
print()
print('Phase 2 signature scoring checkpoint:', 'PASSED' if all_pass else 'INCOMPLETE')
print()
print('Next:  01e_cpa_gene_programs.ipynb — CPA model training + CINEMA-OT')

In [ ]:
from datetime import datetime
import subprocess

timestamp   = datetime.now().strftime('%Y%m%d_%H%M')
report_dir  = Path('../../results/reports')
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / f'01d_signature_scoring_{timestamp}.html'

nb_path = (
    __vsc_ipynb_file__
    if '__vsc_ipynb_file__' in dir()
    else '01d_signature_scoring.ipynb'
)
subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'html', '--output', str(report_path), nb_path],
    check=False,
)
print(f'Report saved to {report_path}')